# ClASP environment smoke test

This notebook verifies that the project environment can import ClASP and run a small analysis. Run all cells from the repository root.

In [1]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd

assert Path("app.py").is_file(), "Start Jupyter from the ClASP repository root."
print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}; NumPy: {np.__version__}")

Python: 3.12.9
pandas: 2.3.3; NumPy: 2.4.4


In [2]:
# Import each project module so missing runtime dependencies fail visibly.
from srcs.utils_features.common import set_paper_plot_style
from srcs.utils_features.data_description import (
    build_continuous_summary_table,
    get_basic_info,
)
from srcs.utils_features.responder_analysis import calculate_swc_te
from srcs.utils_visualisation import (
    common,
    correlation_plots,
    distribution_plots,
    group_comparison_plots,
    regression_analysis,
    render_ols_page,
    render_responder_page,
)

set_paper_plot_style()
print("All ClASP modules imported successfully.")

2026-09-22 14:36:59.994 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-22 14:36:59.995 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


All ClASP modules imported successfully.


In [3]:
sample = pd.DataFrame(
    {
        "participant_id": ["P001", "P002", "P003", "P004", "P005", "P006"],
        "group": ["Control"] * 3 + ["Training"] * 3,
        "age": [45, 52, 38, 41, 49, 44],
        "feature_pre": [32.5, 28.0, 35.1, 30.2, 26.5, 29.4],
        "feature_post": [33.1, 27.6, 35.7, 33.8, 29.0, 31.7],
    }
)

basic_info = get_basic_info(sample)
summary = build_continuous_summary_table(
    sample,
    cont_cols=["age", "feature_pre", "feature_post"],
    group_col="group",
    split_by_group=True,
)

assert len(basic_info) == sample.shape[1]
assert list(summary["variable"]) == ["age", "feature_pre", "feature_post"]
summary

,variable,Full cohort,Control,Training,p_value,cohens_d
0,age,44.83 ± 5.12 (n=6),45.00 ± 7.00 (n=3),44.67 ± 4.04 (n=3),0.9465,0.06
1,feature_pre,30.28 ± 3.11 (n=6),31.87 ± 3.59 (n=3),28.70 ± 1.95 (n=3),0.2506,1.10
2,feature_post,31.82 ± 3.05 (n=6),32.13 ± 4.14 (n=3),31.50 ± 2.41 (n=3),0.8299,0.19


In [4]:
print("ClASP notebook smoke test passed.")

ClASP notebook smoke test passed.


In [5]:
import re 
import pandas as pd 

df = pd.read_excel('/home/sachakra/Documents/Sabyasachi/2-Data-labsexpsTrials/DiKAP/Export_DIKAP_28-10-2026_final.xlsx',
                    sheet_name='data_DIKAP_20261013-134841')
temporal_cols = [
        col for col in df.columns
        if re.search(r'_T\d+|_Tp\d+|_tp\d+', col)
    ]
temporal_cols 
df.columns


import re
from collections import defaultdict

temporal_pattern = re.compile(
    r'^(?P<family>.*?)'
    r'(?:_T[pP]?|[\s_-]+hour[\s_-]*)'
    r'(?P<time>\d+)$',
    flags=re.IGNORECASE
)

temporal_families = defaultdict(list)

for col in df.columns:
    match = temporal_pattern.match(str(col).strip())

    if match:
        family = match.group("family").strip(" _-")
        timepoint = int(match.group("time"))

        temporal_families[family].append((timepoint, col))

# Sort each family chronologically and retain the column names
temporal_families = {
    family: [col for _, col in sorted(columns)]
    for family, columns in temporal_families.items()
    if len(columns) > 1  # Only repeated temporal families
}

temporal_cols = [
    col
    for columns in temporal_families.values()
    for col in columns
]

temporal_cols




import re
import pandas as pd
from collections import defaultdict


# ============================================================
# Temporal column patterns
# ============================================================

TEMPORAL_PATTERNS = [
    # Examples:
    # Spiro_1min_Sys
    # Spiro_3min_HR
    # Spiro_10minutes_Dias
    re.compile(
        r"""
        ^
        (?P<prefix>.*?)
        [\s_-]+
        (?P<time>\d+)
        \s*
        (?P<unit>
            seconds?|secs?|
            minutes?|mins?|
            hours?|hrs?|
            days?|
            weeks?|
            months?|
            years?
        )
        (?P<suffix>[\s_-]+.*)?
        $
        """,
        re.IGNORECASE | re.VERBOSE,
    ),

    # Examples:
    # OCTA - VD Deep layer hour 4
    # OCTA - VD Deep layer hour_4
    # Variable day 2
    re.compile(
        r"""
        ^
        (?P<prefix>.*?)
        [\s_-]+
        (?P<unit>
            seconds?|secs?|
            minutes?|mins?|
            hours?|hrs?|
            days?|
            weeks?|
            months?|
            years?
        )
        [\s_-]*
        (?P<time>\d+)
        (?P<suffix>.*)
        $
        """,
        re.IGNORECASE | re.VERBOSE,
    ),

    # Examples:
    # Variable_T1
    # Variable_T2
    # Variable_Tp1
    # Variable_tp3
    re.compile(
        r"""
        ^
        (?P<prefix>.*?)
        _T[p]?
        (?P<time>\d+)
        (?P<suffix>.*)
        $
        """,
        re.IGNORECASE | re.VERBOSE,
    ),
]


# ============================================================
# Helper functions
# ============================================================

def normalize_time_unit(unit):
    """Convert different unit spellings into one standard form."""

    if unit is None:
        return "timepoint"

    unit = unit.lower()

    unit_mapping = {
        "second": "sec",
        "seconds": "sec",
        "sec": "sec",
        "secs": "sec",

        "minute": "min",
        "minutes": "min",
        "min": "min",
        "mins": "min",

        "hour": "hour",
        "hours": "hour",
        "hr": "hour",
        "hrs": "hour",

        "day": "day",
        "days": "day",

        "week": "week",
        "weeks": "week",

        "month": "month",
        "months": "month",

        "year": "year",
        "years": "year",
    }

    return unit_mapping.get(unit, unit)


def create_family_name(prefix, suffix):
    """
    Reconstruct the family name after removing the temporal part.

    Examples
    --------
    Spiro_3min_HR  -> Spiro_HR
    Spiro_1min_Sys -> Spiro_Sys
    Variable_T1    -> Variable
    """

    prefix = (prefix or "").strip(" _-")
    suffix = suffix or ""

    if not suffix.strip(" _-"):
        return prefix

    # Preserve underscore-style names such as Spiro_HR
    if suffix.lstrip().startswith("_"):
        separator = "_"

    # Preserve hyphen-style names
    elif suffix.lstrip().startswith("-"):
        separator = " - "

    else:
        separator = " "

    suffix = suffix.strip(" _-")

    return f"{prefix}{separator}{suffix}".strip()


def parse_temporal_column(column):
    """
    Parse a column name and return its temporal information.

    Returns None when the column is not recognized as temporal.
    """

    column = str(column).strip()

    for pattern in TEMPORAL_PATTERNS:
        match = pattern.match(column)

        if match:
            values = match.groupdict()

            family = create_family_name(
                prefix=values.get("prefix"),
                suffix=values.get("suffix"),
            )

            return {
                "column": column,
                "family": family,
                "time": int(values["time"]),
                "unit": normalize_time_unit(values.get("unit")),
            }

    return None


def identify_temporal_families(df, minimum_timepoints=1):
    """
    Identify and group temporal columns in a DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame.

    minimum_timepoints : int, default=1
        Minimum number of columns required to retain a family.
        Use 2 to keep only repeated-measurement families.

    Returns
    -------
    temporal_cols : list
        All recognized temporal columns.

    temporal_families : dict
        Dictionary keyed by (family, unit).

    temporal_summary : pandas.DataFrame
        Tabular summary of detected temporal columns.
    """

    detected_columns = []

    for column in df.columns:
        parsed = parse_temporal_column(column)

        if parsed is not None:
            detected_columns.append(parsed)

    # Group by family and temporal unit
    grouped = defaultdict(list)

    for item in detected_columns:
        key = (item["family"], item["unit"])
        grouped[key].append(item)

    # Sort chronologically and filter by family size
    temporal_families = {}

    for key, items in grouped.items():
        items = sorted(
            items,
            key=lambda item: (item["time"], item["column"])
        )

        if len(items) >= minimum_timepoints:
            temporal_families[key] = items

    # Create a flat list of temporal columns
    temporal_cols = [
        item["column"]
        for items in temporal_families.values()
        for item in items
    ]

    # Create a summary DataFrame
    summary_rows = []

    for (family, unit), items in temporal_families.items():
        for item in items:
            summary_rows.append({
                "family": family,
                "unit": unit,
                "time": item["time"],
                "column": item["column"],
                "number_of_timepoints": len(items),
            })

    temporal_summary = pd.DataFrame(summary_rows)

    if not temporal_summary.empty:
        temporal_summary = (
            temporal_summary
            .sort_values(["family", "unit", "time"])
            .reset_index(drop=True)
        )

    return temporal_cols, temporal_families, temporal_summary


# ============================================================
# Run detection on your DataFrame
# ============================================================

# Use minimum_timepoints=1 to show every detected family
temporal_cols, temporal_families, temporal_summary = (
    identify_temporal_families(
        df,
        minimum_timepoints=1,
    )
)


# ============================================================
# Display results
# ============================================================

print("Detected temporal columns:")
for column in temporal_cols:
    print(f"  - {column}")


print("\nDetected temporal families:")

for (family, unit), items in temporal_families.items():
    print(f"\n{family} [{unit}]")

    for item in items:
        print(f"  Time {item['time']}: {item['column']}")


# Convenient table for Jupyter
display(temporal_summary)


Detected temporal columns:
  - Spiro_1min [W]
  - Spiro_3min [W]
  - Spiro_1min_Laktat
  - Spiro_1min_Borg
  - Spiro_3min_Borg
  - Spiro_1min_HR
  - Spiro_3min_HR
  - Spiro_1min_Sys
  - Spiro_3min_Sys
  - Spiro_1min_Dias
  - Spiro_3min_Dias
  - OCTA - VD Superficial layer hour 1
  - OCTA - VD Superficial layer hour 2
  - OCTA - VD Superficial layer hour 3
  - OCTA - VD Superficial layer hour 4
  - OCTA - VD Superficial layer hour 5
  - OCTA - VD Superficial layer hour 6
  - OCTA - VD Superficial layer hour 7
  - OCTA - VD Superficial layer hour 8
  - OCTA - VD Superficial layer hour 9
  - OCTA - VD Superficial layer hour 10
  - OCTA - VD Superficial layer hour 11
  - OCTA - VD Superficial layer hour 12
  - OCTA - Mid layer hour 1
  - OCTA - Mid layer hour 2
  - OCTA - Mid layer hour 3
  - OCTA - Mid layer hour 4
  - OCTA - Mid layer hour 5
  - OCTA - Mid layer hour 6
  - OCTA - Mid layer hour 7
  - OCTA - Mid layer hour 8
  - OCTA - Mid layer hour 9
  - OCTA - Mid layer hour 10
  - OCT

,family,unit,time,column,number_of_timepoints
0,OCTA - Mid layer,hour,1,OCTA - Mid layer hour 1,12
1,OCTA - Mid layer,hour,2,OCTA - Mid layer hour 2,12
2,OCTA - Mid layer,hour,3,OCTA - Mid layer hour 3,12
3,OCTA - Mid layer,hour,4,OCTA - Mid layer hour 4,12
4,OCTA - Mid layer,hour,5,OCTA - Mid layer hour 5,12
5,OCTA - Mid layer,hour,6,OCTA - Mid layer hour 6,12
6,OCTA - Mid layer,hour,7,OCTA - Mid layer hour 7,12
7,OCTA - Mid layer,hour,8,OCTA - Mid layer hour 8,12
8,OCTA - Mid layer,hour,9,OCTA - Mid layer hour 9,12
9,OCTA - Mid layer,hour,10,OCTA - Mid layer hour 10,12


In [8]:
import re
import pandas as pd
from collections import defaultdict


NUMBER_PATTERN = re.compile(r'\d+(?:\.\d+)?')


# def normalize_template(template):
#     """Normalize a template for reliable grouping."""

#     template = re.sub(r'\s+', ' ', template)
#     return template.strip().lower()

def normalize_template(template):
    """
    Normalize spaces, underscores, and hyphens so differently formatted
    columns can still be grouped into the same family.
    """

    template = str(template).strip().lower()

    # Treat spaces, underscores and hyphens as equivalent separators
    template = re.sub(r'[\s_-]+', '_', template)

    # Remove underscores around the time placeholder
    template = re.sub(
        r'_*\{time\}_*',
        '{time}',
        template,
    )

    # Normalize separators around brackets
    template = re.sub(r'_*([\[\]()])_*', r'\1', template)

    return template.strip('_')


def template_to_family_name(template):
    """
    Convert a numbered-column template into a readable family name.

    Examples
    --------
    Spiro_Level{time} [W]              -> Spiro_Level [W]
    Spiro_{time}min_Sys                -> Spiro_Sys
    OCTA - VD Deep layer hour {time}   -> OCTA - VD Deep layer
    Variable_T{time}                   -> Variable
    """

    family = template

    # Remove patterns such as _T1, _Tp2 and _tp3
    family = re.sub(
        r'(?i)[\s_-]*T[p]?[\s_-]*\{time\}',
        '',
        family,
    )

    units = (
        r'seconds?|secs?|'
        r'minutes?|mins?|'
        r'hours?|hrs?|'
        r'days?|'
        r'weeks?|'
        r'months?|'
        r'years?'
    )

    # Remove "1min", "3hours", "2 days", etc.
    family = re.sub(
        rf'(?i)[\s_-]*\{{time\}}\s*(?:{units})',
        '',
        family,
    )

    # Remove "hour 1", "day 3", "week_2", etc.
    family = re.sub(
        rf'(?i)[\s_-]*(?:{units})[\s_-]*\{{time\}}',
        '',
        family,
    )

    # For patterns such as Level1, Visit2 or Stage3, retain the
    # descriptive word and remove only the number.
    family = family.replace('{time}', '')

    # Clean separators produced by removing the temporal component
    family = re.sub(r'_{2,}', '_', family)
    family = re.sub(r'\s{2,}', ' ', family)
    family = re.sub(r'[-_]\s*$', '', family)
    family = re.sub(r'^\s*[-_]', '', family)

    # Remove separators immediately before brackets
    family = re.sub(r'[_-]+\s*(?=\[)', ' ', family)

    return family.strip(' _-')


def generate_number_templates(column):
    """
    Generate one candidate template for every number in a column.

    This also supports columns containing other unrelated numbers.
    """

    column = str(column).strip()
    candidates = []

    for number_position, match in enumerate(NUMBER_PATTERN.finditer(column)):
        time_string = match.group()

        template = (
            column[:match.start()]
            + '{time}'
            + column[match.end():]
        )

        candidates.append({
            'column': column,
            'template': template,
            'normalized_template': normalize_template(template),
            'time': float(time_string),
            'time_text': time_string,
            'number_position': number_position,
        })

    return candidates


def identify_repeated_measure_families(
    df,
    minimum_timepoints=2,
):
    """
    Automatically detect repeated-measure column families.

    A family is identified when two or more columns become identical
    after replacing one numerical value with {time}.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame whose columns will be examined.

    minimum_timepoints : int, default=2
        Minimum number of distinct numerical timepoints required for
        a family to be retained.

    Returns
    -------
    repeated_columns : list
        Flat list containing all repeated-measure columns.

    families : dict
        Dictionary containing detected families.

    summary : pandas.DataFrame
        Summary table with family, timepoint and original column.
    """

    template_candidates = defaultdict(list)

    # Create templates from every numerical part of every column
    for column in df.columns:
        for candidate in generate_number_templates(column):
            key = candidate['normalized_template']
            template_candidates[key].append(candidate)

    families = {}

    for candidates in template_candidates.values():
        # Remove duplicate column/time combinations
        unique_candidates = {}

        for candidate in candidates:
            key = (
                candidate['column'],
                candidate['time'],
                candidate['number_position'],
            )
            unique_candidates[key] = candidate

        candidates = list(unique_candidates.values())

        distinct_columns = {
            candidate['column']
            for candidate in candidates
        }

        distinct_times = {
            candidate['time']
            for candidate in candidates
        }

        # A real repeated family needs multiple columns and timepoints
        if (
            len(distinct_columns) < minimum_timepoints
            or len(distinct_times) < minimum_timepoints
        ):
            continue

        candidates = sorted(
            candidates,
            key=lambda item: (
                item['time'],
                item['column'],
            ),
        )

        original_template = candidates[0]['template']
        family_name = template_to_family_name(original_template)

        # Include the template in the key so that similarly named
        # families cannot overwrite one another.
        family_key = (
            family_name,
            normalize_template(original_template),
        )

        families[family_key] = {
            'family': family_name,
            'template': original_template,
            'timepoints': [
                candidate['time']
                for candidate in candidates
            ],
            'columns': [
                candidate['column']
                for candidate in candidates
            ],
            'items': candidates,
        }

    # Flat list of all detected repeated-measure columns
    repeated_columns = []

    for family_data in families.values():
        repeated_columns.extend(family_data['columns'])

    # Remove duplicates while preserving order
    repeated_columns = list(dict.fromkeys(repeated_columns))

    # Build summary table
    summary_rows = []

    for family_data in families.values():
        for candidate in family_data['items']:
            summary_rows.append({
                'family': family_data['family'],
                'template': family_data['template'],
                'timepoint': candidate['time'],
                'timepoint_text': candidate['time_text'],
                'column': candidate['column'],
                'number_of_timepoints': len(
                    set(family_data['timepoints'])
                ),
            })

    summary = pd.DataFrame(summary_rows)

    if not summary.empty:
        summary = (
            summary
            .sort_values(
                ['family', 'template', 'timepoint']
            )
            .reset_index(drop=True)
        )

    return repeated_columns, families, summary


temporal_cols, temporal_families, temporal_summary = (
    identify_repeated_measure_families(
        df,
        minimum_timepoints=2,
    )
)

display(temporal_summary)

for family_data in temporal_families.values():
    print(f"\nFamily: {family_data['family']}")
    print(f"Template: {family_data['template']}")

    for item in family_data['items']:
        print(
            f"  {item['time_text']}: "
            f"{item['column']}"
        )

,family,template,timepoint,timepoint_text,column,number_of_timepoints
0,Grip_Strength_DG_left [kg],Grip_Strength_DG{time}_left [kg],1.0,1,Grip_Strength_DG1_left [kg],3
1,Grip_Strength_DG_left [kg],Grip_Strength_DG{time}_left [kg],2.0,2,Grip_Strength_DG2_left [kg],3
2,Grip_Strength_DG_left [kg],Grip_Strength_DG{time}_left [kg],3.0,3,Grip_Strength_DG3_left [kg],3
3,Grip_Strength_DG_right [kg],Grip_Strength_DG{time}_right [kg],1.0,1,Grip_Strength_DG1_right [kg],3
4,Grip_Strength_DG_right [kg],Grip_Strength_DG{time}_right [kg],2.0,2,Grip_Strength_DG2_right [kg],3
...,...,...,...,...,...,...
130,Wortliste DG_Roh,Wortliste DG{time}_Roh,2.0,2,Wortliste DG2_Roh,3
131,Wortliste DG_Roh,Wortliste DG{time}_Roh,3.0,3,Wortliste DG3_Roh,3
132,Wortliste DG_z,Wortliste DG{time}_z,1.0,1,Wortliste DG1_z,3
133,Wortliste DG_z,Wortliste DG{time}_z,2.0,2,Wortliste DG2_z,3



Family: Sonstige Medis_Wirkstoff
Template: Sonstige Medis{time}_Wirkstoff
  1: Sonstige Medis1_Wirkstoff
  2: Sonstige Medis2_Wirkstoff
  3: Sonstige Medis3_Wirkstoff
  4: Sonstige Medis4_Wirkstoff
  5: Sonstige Medis5_Wirkstoff
  6: Sonstige Medis6_Wirkstoff
  7: Sonstige Medis7_Wirkstoff
  8: Sonstige Medis8_Wirkstoff
  9: Sonstige Medis9_Wirkstoff
  10: Sonstige Medis10_Wirkstoff
  11: Sonstige Medis11_Wirkstoff

Family: Sonst.Medi_Dosis
Template: Sonst.Medi{time}_Dosis
  1: Sonst.Medi1_Dosis
  2: Sonst.Medi2_Dosis
  3: Sonst.Medi3_Dosis
  4: Sonst.Medi4_Dosis
  5: Sonst.Medi5_Dosis
  6: Sonst.Medi6_Dosis
  7: Sonst.Medi7_Dosis
  8: Sonst.Medi8_Dosis
  9: Sonst.Medi9_Dosis
  10: Sonst.Medi10_Dosis
  11: Sonst.Medi11_Dosis

Family: Wortliste DG_Roh
Template: Wortliste DG{time}_Roh
  1: Wortliste DG1_Roh
  2: Wortliste DG2_Roh
  3: Wortliste DG3_Roh

Family: Wortliste DG_z
Template: Wortliste DG{time}_z
  1: Wortliste DG1_z
  2: Wortliste DG2_z
  3: Wortliste DG3_z

Family: VLMT DG
T